# FaceProof 10 — figures (Day 10)

Generate the report figures from `results.csv` + saved calibration arrays:
1. system diagram, 2. cross-generator AUROC bars, 3. robustness curve, 4. reliability diagrams.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/Ezed9/faceProof.git"
BRANCH   = "day1-preprocess-notebook"   # change to "main" after merge
!git clone -b $BRANCH $REPO_URL
%cd faceProof
!pip install -q pandas matplotlib
import sys, os
sys.path.append(os.getcwd())

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np, pandas as pd
drive.mount('/content/drive')
ROOT          = Path('/content/drive/MyDrive/faceproof')
CROPS_ROOT    = ROOT / 'data' / 'crops'
FEATURES_ROOT = ROOT / 'models' / 'features'
PROBES_ROOT   = ROOT / 'models' / 'probes'
REPORTS_ROOT  = ROOT / 'reports'
MANIFEST      = ROOT / 'data' / 'manifest.csv'
RESULTS_CSV   = REPORTS_ROOT / 'results.csv'
(REPORTS_ROOT / 'figures').mkdir(parents=True, exist_ok=True)

In [ ]:
import matplotlib.pyplot as plt
FIG = REPORTS_ROOT/'figures'; FIG.mkdir(parents=True, exist_ok=True)
r = pd.read_csv(RESULTS_CSV)

## 2. System diagram (schematic)

In [ ]:
fig,ax=plt.subplots(figsize=(10,2.4)); ax.axis('off')
steps=['Face image','Align +\ncompression match','Frozen backbone\n(CLIP / ResNet)','Logistic probe','real / synthetic\n+ calibrated conf.']
for i,s in enumerate(steps):
    ax.add_patch(plt.Rectangle((i*2,0),1.7,1.2,fill=True,fc='#e8eef7',ec='#33548c'))
    ax.text(i*2+0.85,0.6,s,ha='center',va='center',fontsize=9)
    if i<len(steps)-1: ax.annotate('',xy=(i*2+2,0.6),xytext=(i*2+1.7,0.6),arrowprops=dict(arrowstyle='->'))
ax.set_xlim(-0.2,len(steps)*2); ax.set_ylim(-0.2,1.4)
fig.savefig(FIG/'system_diagram.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. Cross-generator AUROC bars

In [ ]:
auc = r[(r['metric']=='AUROC')&(r['corruption']=='none')&(r['seed']==13)]
piv = auc.pivot_table(index='test_gen',columns='condition',values='value',aggfunc='last')
ax = piv.plot(kind='bar', figsize=(9,4.5)); ax.set_ylabel('AUROC'); ax.set_ylim(0.4,1.0)
ax.set_title('AUROC by condition and test generator'); ax.axhline(0.5,ls='--',c='gray')
plt.tight_layout(); plt.savefig(FIG/'crossgen_auroc.png',dpi=150,bbox_inches='tight'); plt.show()

## 4. Robustness curve (JPEG)

In [ ]:
jp = r[(r['corruption']=='jpeg')&(r['metric']=='AUROC')]
if len(jp):
    fig,ax=plt.subplots(figsize=(7,4.5))
    for (cond,tg),g in jp.groupby(['condition','test_gen']):
        g=g.astype({'strength':float}).sort_values('strength',ascending=False)
        ax.plot(g['strength'],g['value'],marker='o',label=f'{cond}/{tg}')
    ax.invert_xaxis(); ax.set_xlabel('JPEG quality'); ax.set_ylabel('AUROC')
    ax.legend(fontsize=8); ax.grid(alpha=.3); plt.tight_layout()
    plt.savefig(FIG/'robustness_jpeg.png',dpi=150,bbox_inches='tight'); plt.show()
else:
    print('No JPEG rows yet — run notebook 04 first.')

## 5. Reliability diagrams (from saved calibration arrays)

In [ ]:
npz_path = REPORTS_ROOT/'calib_arrays.npz'
if npz_path.exists():
    d = np.load(npz_path)
    keys = sorted({k.rsplit('__',1)[0] for k in d.files})
    fig,axes=plt.subplots(1,len(keys),figsize=(4*len(keys),4),squeeze=False)
    for j,key in enumerate(keys):
        yy=d[f'{key}__y']; p=d[f'{key}__p_un']
        bins=np.linspace(0,1,16); xs=[];ys=[]
        for lo,hi in zip(bins[:-1],bins[1:]):
            m=(p>lo)&(p<=hi)
            if m.sum()==0: continue
            xs.append(p[m].mean()); ys.append(yy[m].mean())
        ax=axes[0,j]; ax.plot([0,1],[0,1],'--',c='gray'); ax.plot(xs,ys,marker='o')
        ax.set_title(key,fontsize=8); ax.set_xlabel('confidence'); ax.set_ylabel('accuracy')
    plt.tight_layout(); plt.savefig(FIG/'reliability.png',dpi=150,bbox_inches='tight'); plt.show()
else:
    print('Run notebook 06 first to produce calib_arrays.npz')
print('✅ Day 10 gate: figures saved to reports/figures/')